# F02-P3 Threat

**Ecosystem Threat: Dryland Forest, Mangrove, Peatland**

Showcasing the disturbance in three different ecosystems, mainly on forest disturbances but not limited to hydrological disturbances for peatland ecosystems. F02-P3 Threat answers “where is the most disturbed area and its drivers”.

> **Not runnable yet.** All analysis logic below is real Python. Only file access is stubbed.

**Status.** Complete as scoped: All ecosystems section.

**Known limit of that scope.** Forest disturbance is an internal agreement term for forest degradation with the structural change across 10 years. However, the 10 years of forest degradation is too long, the next v3.1 this will be revised into annual or maximum 5 years analysis.

## Setup

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio

from pyproj import Geod
from rasterio.mask import raster_geometry_mask
from rasterio.mask import mask
from scipy.ndimage import distance_transform_edt

---
## 7.1 All Ecosystem (Overview)

Reports the total ecosystem and disturbed area across three different ecosystem in hectare.

**Data.** `forest_disturbance_v3.tif: any pixel > 0 = disturbed` following C. Bourgoin, 2024. The 
methodology published following JRC-TMF data on degraded and undisturbed forest. Scene approach
focusses on structural decline on forest using TCC and TCH by SIGnal forest cover. However, the drivers
of degradation only limited to selective logging and forest fire

**Calibration warning.** The 0 to 3 values are calibrated on the pooled SEA distribution, so
they are not one to one with the published JRC-TMF. 
**Decisions locked.**

Structural decline within retained forest, assessed as canopy height deficit
relative to an undisturbed reference population defined at >=120 m from any
disturbed forest. This addresses the growing stock and biomass marker of FAO
(2011, FRA Working Paper 177). It is not a complete assessment of forest
degradation as defined by FAO, and no globally agreed operational definition
currently exists. Detection threshold set at the 5th percentile of reference
population height change, giving a nominal 5% false-positive rate. Results
are reported as area statistics by stratum; per-pixel interpretation is not
supported at 30 m given a product RMSE of 6.6-9.1 m.

**Example render**

## All Ecosystem Screening

| Summary | Value |
|---|---:|
| **Total ecosystem area** | **81,880.23 ha** |
| **Total disturbed area** | **7,383.64 ha (9.02%)** |

### Ecosystem Breakdown

| Ecosystem | Area (ha) | % Total | Disturbed (ha) | Disturbed % |
|---|---:|---:|---:|---:|
| **Dryland forest** | 81,880.23 | 100.00% | 7,383.64 | 9.02% |
| **Mangrove** | 0.00 | 0.00% | 0.00 | 0.00% |
| **Peatland** | 0.00 | 0.00% | 0.00 | 0.00% |
| **Other** | 0.00 | 0.00% | 0.00 | 0.00% |

In [ ]:
def pixel_area_by_row(transform, height):
    """
    Calculate pixel area in hectares by raster row.
    Suitable for EPSG:4326.
    """

    pixel_width = abs(transform.a)

    row_areas = np.zeros(height)

    for row in range(height):

        north = transform.f + row * transform.e
        south = north + transform.e

        west = transform.c
        east = west + pixel_width

        area_m2, _ = GEOD.polygon_area_perimeter(
            [west, east, east, west],
            [north, north, south, south]
        )

        row_areas[row] = abs(area_m2) / 10000

    return row_areas


def calculate_mask_area_ha(binary_mask, transform):
    """
    Calculate area of True pixels in hectares.
    """

    row_areas = pixel_area_by_row(
        transform,
        binary_mask.shape[0]
    )

    pixels_per_row = binary_mask.sum(axis=1)

    return float(
        np.sum(
            pixels_per_row * row_areas
        )
    )


def read_masked_raster(
    raster_path,
    aoi_geometry,
    aoi_crs
):
    """
    Read only the AOI window from a raster.
    """

    with rasterio.open(raster_path) as src:

        if src.crs is None:
            raise ValueError(
                f"Raster has no CRS: {raster_path}"
            )

        if src.crs != aoi_crs:

            raster_aoi = (
                gpd.GeoSeries(
                    [aoi_geometry],
                    crs=aoi_crs
                )
                .to_crs(src.crs)
                .iloc[0]
            )

        else:

            raster_aoi = aoi_geometry


        outside_mask, transform, window = (
            raster_geometry_mask(
                src,
                [raster_aoi.__geo_interface__],
                crop=True,
                all_touched=False
            )
        )


        data = src.read(
            1,
            window=window,
            masked=True
        )


        valid_mask = (
            ~outside_mask
            & ~np.ma.getmaskarray(data)
        )


        return (
            data.data,
            valid_mask,
            transform,
            src.crs
        )


# =============================================================================
# Main Analysis
# =============================================================================

def analyze_all_ecosystem():

    # -------------------------------------------------------------------------
    # 1. Read AOI
    # -------------------------------------------------------------------------

    aoi = gpd.read_file(
        USER_AOI
    )

    if aoi.empty:
        raise ValueError(
            "AOI contains no features."
        )

    if aoi.crs is None:
        raise ValueError(
            "AOI has no CRS."
        )


    valid_geometry = aoi.geometry[
        aoi.geometry.notna()
        & ~aoi.geometry.is_empty
    ]


    if valid_geometry.empty:
        raise ValueError(
            "AOI contains no valid geometry."
        )


    aoi_geometry = (
        valid_geometry.union_all()
    )


    # -------------------------------------------------------------------------
    # 2. Read Ecosystem Raster
    # -------------------------------------------------------------------------

    (
        ecosystem_data,
        ecosystem_valid,
        ecosystem_transform,
        ecosystem_crs
    ) = read_masked_raster(
        ECOSYSTEM,
        aoi_geometry,
        aoi.crs
    )


    # -------------------------------------------------------------------------
    # 3. Read Disturbance Raster
    # -------------------------------------------------------------------------

    (
        disturbance_data,
        disturbance_valid,
        disturbance_transform,
        disturbance_crs
    ) = read_masked_raster(
        DISTURBANCE,
        aoi_geometry,
        aoi.crs
    )


    # -------------------------------------------------------------------------
    # IMPORTANT:
    # Both rasters must share the same clipped grid for direct boolean masking.
    # -------------------------------------------------------------------------

    if ecosystem_data.shape != disturbance_data.shape:

        raise ValueError(
            "Ecosystem and disturbance raster grids do not match "
            "after clipping."
        )


    if not np.allclose(
        ecosystem_transform,
        disturbance_transform
    ):

        raise ValueError(
            "Ecosystem and disturbance raster transforms do not match."
        )


    # -------------------------------------------------------------------------
    # 4. Total Ecosystem Mask
    # -------------------------------------------------------------------------

    total_ecosystem_mask = (
        ecosystem_valid
        & np.isin(
            ecosystem_data,
            list(
                ECOSYSTEM_CLASSES.keys()
            )
        )
    )


    total_ecosystem_area_ha = (
        calculate_mask_area_ha(
            total_ecosystem_mask,
            ecosystem_transform
        )
    )


    # -------------------------------------------------------------------------
    # 5. Total Disturbed Area
    # -------------------------------------------------------------------------

    disturbance_mask = (
        disturbance_valid
        & (disturbance_data > 0)
    )


    total_disturbed_mask = (
        total_ecosystem_mask
        & disturbance_mask
    )


    total_disturbed_area_ha = (
        calculate_mask_area_ha(
            total_disturbed_mask,
            ecosystem_transform
        )
    )


    total_disturbed_percentage = (
        total_disturbed_area_ha
        / total_ecosystem_area_ha
        * 100
        if total_ecosystem_area_ha > 0
        else 0
    )


    # -------------------------------------------------------------------------
    # 6. Ecosystem Breakdown
    # -------------------------------------------------------------------------

    ecosystem_results = {}


    for class_value, class_name in (
        ECOSYSTEM_CLASSES.items()
    ):

        ecosystem_mask = (
            ecosystem_valid
            & (ecosystem_data == class_value)
        )


        ecosystem_area_ha = (
            calculate_mask_area_ha(
                ecosystem_mask,
                ecosystem_transform
            )
        )


        ecosystem_percentage = (
            ecosystem_area_ha
            / total_ecosystem_area_ha
            * 100
            if total_ecosystem_area_ha > 0
            else 0
        )


        # ---------------------------------------------------------------------
        # Disturbed within ecosystem
        # ---------------------------------------------------------------------

        ecosystem_disturbed_mask = (
            ecosystem_mask
            & disturbance_mask
        )


        ecosystem_disturbed_area_ha = (
            calculate_mask_area_ha(
                ecosystem_disturbed_mask,
                ecosystem_transform
            )
        )


        ecosystem_disturbed_percentage = (
            ecosystem_disturbed_area_ha
            / ecosystem_area_ha
            * 100
            if ecosystem_area_ha > 0
            else 0
        )


        ecosystem_results[
            class_name
        ] = {

            "area_ha":
                round(
                    ecosystem_area_ha,
                    2
                ),

            "percentage_total":
                round(
                    ecosystem_percentage,
                    2
                ),

            "disturbed_area_ha":
                round(
                    ecosystem_disturbed_area_ha,
                    2
                ),

            "disturbed_percentage":
                round(
                    ecosystem_disturbed_percentage,
                    2
                )
        }


    # -------------------------------------------------------------------------
    # Return
    # -------------------------------------------------------------------------

    return {

        "total_ecosystem_area_ha":
            round(
                total_ecosystem_area_ha,
                2
            ),

        "total_disturbed_area_ha":
            round(
                total_disturbed_area_ha,
                2
            ),

        "total_disturbed_percentage":
            round(
                total_disturbed_percentage,
                2
            ),

        "ecosystems":
            ecosystem_results
    }


# =============================================================================
# Run
# =============================================================================

result = analyze_all_ecosystem()


# =============================================================================
# Display
# =============================================================================

print()
print("=" * 70)
print("ALL ECOSYSTEM SCREENING")
print("=" * 70)

print(
    f"Total ecosystem area : "
    f"{result['total_ecosystem_area_ha']:,.2f} ha"
)

print(
    f"Total disturbed area : "
    f"{result['total_disturbed_area_ha']:,.2f} ha "
    f"({result['total_disturbed_percentage']:.2f}%)"
)


print()
print("Ecosystem breakdown:")
print("-" * 90)

print(
    f"{'Ecosystem':<20}"
    f"{'Area (ha)':>15}"
    f"{'% Total':>12}"
    f"{'Disturbed (ha)':>20}"
    f"{'Disturbed %':>15}"
)

print("-" * 90)


for ecosystem_name, values in (
    result["ecosystems"].items()
):

    print(
        f"{ecosystem_name:<20}"
        f"{values['area_ha']:>15,.2f}"
        f"{values['percentage_total']:>11.2f}%"
        f"{values['disturbed_area_ha']:>20,.2f}"
        f"{values['disturbed_percentage']:>14.2f}%"
    )

---
## 7.2 Dryland Forest Ecosystem

Reports the drivers of dryland forest disturbance.

**Data.** `forest_drivers_v3.tif: pixel ranging from 1 to 11 ` This data comes from
Bart Slagter, et al 2026 https://doi.org/10.21203/rs.3.rs-7424252/v1
Which only focuses on classifying the key drivers of forest disturbances and 
may not include all potential causes of deforestation.


**Calibration warning.** Post-processing steps identified where fire coincided with the clearing of agricultural land, based on the presence of VIIRS fire alerts (within a 500 m buffer around the alert) and a low post-disturbance normalized-burn ration in the following month’s Sentinen-2 composite.
Post-processing steps masking out the area outside forest_disturbance_v3.tif and re-calibrate
with disaster risk from ADPC. However, this data only consider high and very high disaster risk
as part of forest disturbance.

**Example render**

## Dryland Forest

| Indicator | Value |
|---|---:|
| **Total area** | **81,880.23 ha** |
| **Remaining forest** | **74,824.43 ha (91.38%)** |
| **Disturbed** | **5,572.79 ha (6.81%)** |
| **Forest loss** | **3,095.78 ha (3.78%)** |
| **Forest gain** | **472.14 ha (0.58%)** |

### Non-natural Drivers

- Small-scale agriculture
- Small-scale agriculture (fire)
- Large-scale agriculture
- Large-scale agriculture (fire)
- Road development
- Selective logging
- Mining

### Natural Drivers

- Flooding
- Forest fire
- Landslide

### Other Drivers

- Non-productive conversion
- Unknown

In [ ]:
from config import (
    AOI,
    ECOSYSTEM,
    HISTORICAL,
    FOREST_2024,
    DISTURBANCE,
    FOREST_GAIN,
    FOREST_DRIVERS,
    DRIVERS_DISTURBANCE,
    FLOOD_RISK,
    LANDSLIDE_RISK,
    STORM_RISK,
    DRYLAND,
    REMAINING_FOREST,
    FOREST_LOSS,
    CURRENT_FOREST,
    FOREST_GAIN_VALUE,
    FOREST_DRIVER_CLASSES,
    NATURAL_DRIVERS,
)

GEOD = Geod(ellps="WGS84")

# =============================================================================
# Read AOI
# =============================================================================

aoi = gpd.read_file(AOI)

if aoi.empty:
    raise ValueError("AOI contains no features.")

if aoi.crs is None:
    raise ValueError("AOI has no CRS.")


geometry = [
    geom.__geo_interface__
    for geom in aoi.geometry
    if geom is not None and not geom.is_empty
]


# =============================================================================
# Read raster clipped to AOI
# =============================================================================

def read(path):

    with rasterio.open(path) as src:

        shapes = geometry

        if aoi.crs != src.crs:

            projected = aoi.to_crs(
                src.crs
            )

            shapes = [
                geom.__geo_interface__
                for geom in projected.geometry
                if geom is not None
                and not geom.is_empty
            ]

        data, transform = mask(
            src,
            shapes,
            crop=True,
            filled=False
        )

        return (
            data[0],
            transform
        )


# =============================================================================
# Pixel area for EPSG:4326
# =============================================================================

GEOD = Geod(
    ellps="WGS84"
)


def area_ha(mask_array, transform):

    mask_array = np.ma.filled(
        mask_array,
        False
    )

    total = 0.0

    for row in range(mask_array.shape[0]):

        count = np.count_nonzero(
            mask_array[row]
        )

        if count == 0:
            continue

        north = transform.f + row * transform.e
        south = north + transform.e

        west = transform.c
        east = west + transform.a

        pixel_m2, _ = GEOD.polygon_area_perimeter(
            [west, east, east, west],
            [north, north, south, south]
        )

        total += count * abs(pixel_m2)

    return total / 10000


# =============================================================================
# Load core rasters
# =============================================================================

eco, transform = read(
    ECOSYSTEM
)

historical, _ = read(
    HISTORICAL
)

forest2024, _ = read(
    FOREST_2024
)

disturbance, _ = read(
    DISTURBANCE
)

gain, _ = read(
    FOREST_GAIN
)

drivers, _ = read(
    FOREST_DRIVERS
)


# =============================================================================
# Core masks
# =============================================================================

dryland = (
    eco == DRYLAND
)


remaining = (
    dryland
    & (
        historical
        == REMAINING_FOREST
    )
)


forest_loss = (
    dryland
    & (
        historical
        == FOREST_LOSS
    )
)


current_forest = (
    dryland
    & (
        forest2024
        == CURRENT_FOREST
    )
)


disturbed = (
    current_forest
    & (
        disturbance > 0
    )
)


forest_gain = (
    dryland
    & (
        gain
        == FOREST_GAIN_VALUE
    )
)


# =============================================================================
# Areas
# =============================================================================

total_area = area_ha(
    dryland,
    transform
)

remaining_area = area_ha(
    remaining,
    transform
)

disturbed_area = area_ha(
    disturbed,
    transform
)

loss_area = area_ha(
    forest_loss,
    transform
)

gain_area = area_ha(
    forest_gain,
    transform
)


# =============================================================================
# Percentages
# =============================================================================

def percentage(area):

    if total_area == 0:
        return 0

    return (
        area
        / total_area
        * 100
    )


# =============================================================================
# Non-natural + explicit other forest drivers
# =============================================================================

driver_values = np.unique(
    drivers[disturbed]
)


non_natural = []
other = []


for value in driver_values:

    # Ignore masked/nodata values
    if np.ma.is_masked(value):
        continue

    value = int(value)

    if value not in FOREST_DRIVER_CLASSES:
        continue


    driver_name = (
        FOREST_DRIVER_CLASSES[
            value
        ]
    )


    if (
        driver_name
        == "Non-productive conversion"
    ):

        other.append(
            driver_name
        )

    else:

        non_natural.append(
            driver_name
        )


# =============================================================================
# Natural drivers
# =============================================================================

natural = []

# Used later so pixels explained by natural drivers
# are not classified as Unknown.
natural_presence = np.zeros(
    disturbed.shape,
    dtype=bool
)


for (
    driver_name,
    config
) in NATURAL_DRIVERS.items():

    driver_raster, _ = read(
        config["raster"]
    )


    driver_mask = (
        disturbed
        & np.isin(
            driver_raster,
            config["values"]
        )
    )


    if np.any(
        driver_mask
    ):

        natural.append(
            driver_name
        )

        natural_presence |= (
            driver_mask
        )


# =============================================================================
# Unknown driver
# =============================================================================

known_forest_driver = (
    disturbed
    & np.isin(
        drivers,
        list(
            FOREST_DRIVER_CLASSES.keys()
        )
    )
)


unknown = (
    disturbed
    & ~known_forest_driver
    & ~natural_presence
)


if np.any(
    unknown
):

    other.append(
        "Unknown"
    )


# =============================================================================
# Backend result
# =============================================================================

result = {

    "total_area_ha":
        round(
            total_area,
            2
        ),

    "remaining_forest": {
        "area_ha":
            round(
                remaining_area,
                2
            ),

        "percentage":
            round(
                percentage(
                    remaining_area
                ),
                2
            )
    },

    "disturbed": {
        "area_ha":
            round(
                disturbed_area,
                2
            ),

        "percentage":
            round(
                percentage(
                    disturbed_area
                ),
                2
            )
    },

    "forest_loss": {
        "area_ha":
            round(
                loss_area,
                2
            ),

        "percentage":
            round(
                percentage(
                    loss_area
                ),
                2
            )
    },

    "forest_gain": {
        "area_ha":
            round(
                gain_area,
                2
            ),

        "percentage":
            round(
                percentage(
                    gain_area
                ),
                2
            )
    },

    "drivers": {

        "non_natural":
            non_natural,

        "natural":
            natural,

        "other":
            other
    }
}


# =============================================================================
# Output
# =============================================================================

print()
print("DRYLAND FOREST")
print("-" * 70)


print(
    f"Total area       : "
    f"{result['total_area_ha']:,.2f} ha"
)


print(
    f"Remaining forest : "
    f"{result['remaining_forest']['area_ha']:,.2f} ha "
    f"({result['remaining_forest']['percentage']:.2f}%)"
)


print(
    f"Disturbed        : "
    f"{result['disturbed']['area_ha']:,.2f} ha "
    f"({result['disturbed']['percentage']:.2f}%)"
)


print(
    f"Forest loss      : "
    f"{result['forest_loss']['area_ha']:,.2f} ha "
    f"({result['forest_loss']['percentage']:.2f}%)"
)


print(
    f"Forest gain      : "
    f"{result['forest_gain']['area_ha']:,.2f} ha "
    f"({result['forest_gain']['percentage']:.2f}%)"
)


# =============================================================================
# Driver output
# =============================================================================

print()
print("Non-natural drivers:")

if non_natural:

    for driver in non_natural:
        print(
            f"  - {driver}"
        )

else:

    print(
        "  None detected"
    )


print()
print("Natural drivers:")

if natural:

    for driver in natural:
        print(
            f"  - {driver}"
        )

else:

    print(
        "  None detected"
    )


print()
print("Other drivers:")

if other:

    for driver in other:
        print(
            f"  - {driver}"
        )

else:

    print(
        "  None detected"
    )

---
## 3.3 Mangrove Ecosystem

Reports the drivers of mangrove disturbance.

**Data.** `forest_drivers_v3.tif: pixel ranging from 1 to 11 ` This data comes from
Bart Slagter, et al 2026 https://doi.org/10.21203/rs.3.rs-7424252/v1
Which only focuses on classifying the key drivers of forest disturbances and 
may not include all potential causes of deforestation.


**Calibration warning.** Post-processing steps identified where fire coincided with the clearing of agricultural land, based on the presence of VIIRS fire alerts (within a 500 m buffer around the alert) and a low post-disturbance normalized-burn ration in the following month’s Sentinen-2 composite.
Post-processing steps masking out the area outside forest_disturbance_v3.tif and re-calibrate
with disaster risk from ADPC. However, this data only consider high and very high disaster risk
as part of forest disturbance.

## Mangrove Disturbance

| Indicator | Value |
|---|---:|
| **Total area** | **100,810.15 ha** |
| **Remaining mangrove forest** | **55,491.97 ha (55.05%)** |
| **Disturbed** | **150.89 ha (0.15%)** |
| **Main pressure** | **Not identified** |

### Mangrove Disturbance Drivers

#### Non-natural Drivers

None detected.

#### Natural Drivers

None detected.

#### Other Drivers

- Other

In [ ]:

aoi = gpd.read_file(
    USER_AOI
)

if aoi.empty:
    raise ValueError(
        "AOI contains no features."
    )

if aoi.crs is None:
    raise ValueError(
        "AOI has no CRS."
    )

aoi = aoi[
    aoi.geometry.notna()
    & ~aoi.geometry.is_empty
].copy()

if aoi.empty:
    raise ValueError(
        "AOI contains no valid geometry."
    )


# =============================================================================
# READ RASTER CLIPPED TO USER AOI
# =============================================================================

def read_raster(
    raster_path
):

    with rasterio.open(
        raster_path
    ) as src:

        if src.crs is None:
            raise ValueError(
                f"Raster has no CRS: {raster_path}"
            )

        polygon = (
            aoi.to_crs(src.crs)
            if aoi.crs != src.crs
            else aoi
        )

        geometries = [
            geom.__geo_interface__
            for geom in polygon.geometry
        ]

        data, transform = mask(
            src,
            geometries,
            crop=True,
            filled=False
        )

        return (
            data[0],
            transform,
            src.crs
        )


# =============================================================================
# CHECK GRID ALIGNMENT
# =============================================================================

def check_grid(
    reference,
    reference_transform,
    data,
    transform,
    name
):

    if reference.shape != data.shape:

        raise ValueError(
            f"Grid size mismatch: {name}"
        )

    if not np.allclose(
        reference_transform,
        transform
    ):

        raise ValueError(
            f"Grid alignment mismatch: {name}"
        )


# =============================================================================
# AREA CALCULATION
# EPSG:4326 -> geodesic hectare
# =============================================================================

GEOD = Geod(
    ellps="WGS84"
)


def area_ha(
    binary_mask,
    transform
):

    # Avoid MaskError from masked arrays
    binary_mask = np.ma.filled(
        binary_mask,
        False
    ).astype(bool)

    total_m2 = 0.0

    pixel_width = abs(
        transform.a
    )

    for row in range(
        binary_mask.shape[0]
    ):

        pixel_count = np.count_nonzero(
            binary_mask[row]
        )

        if pixel_count == 0:
            continue

        north = (
            transform.f
            + row * transform.e
        )

        south = (
            north
            + transform.e
        )

        west = transform.c
        east = west + pixel_width

        pixel_area_m2, _ = (
            GEOD.polygon_area_perimeter(
                [west, east, east, west],
                [north, north, south, south]
            )
        )

        total_m2 += (
            pixel_count
            * abs(pixel_area_m2)
        )

    return (
        total_m2 / 10000
    )


# =============================================================================
# LOAD RASTERS
# =============================================================================

ecosystem, transform, crs = (
    read_raster(
        ECOSYSTEM
    )
)

historical, historical_transform, _ = (
    read_raster(
        HISTORICAL
    )
)

forest2024, forest2024_transform, _ = (
    read_raster(
        FOREST_2024
    )
)

disturbance, disturbance_transform, _ = (
    read_raster(
        DISTURBANCE
    )
)

drivers, drivers_transform, _ = (
    read_raster(
        DRIVERS
    )
)

storm, storm_transform, _ = (
    read_raster(
        STORM_RISK
    )
)


# =============================================================================
# VALIDATE GRIDS
# =============================================================================

check_grid(
    ecosystem,
    transform,
    historical,
    historical_transform,
    "historical_deforestation_v3.tif"
)

check_grid(
    ecosystem,
    transform,
    forest2024,
    forest2024_transform,
    "forest_2024_v3.tif"
)

check_grid(
    ecosystem,
    transform,
    disturbance,
    disturbance_transform,
    "forest_disturbance_v3.tif"
)

check_grid(
    ecosystem,
    transform,
    drivers,
    drivers_transform,
    "drivers_disturbance_v3.tif"
)

check_grid(
    ecosystem,
    transform,
    storm,
    storm_transform,
    "risk_storm_v3.tif"
)


# =============================================================================
# VALID PIXELS
# =============================================================================

ecosystem_valid = (
    ~np.ma.getmaskarray(
        ecosystem
    )
)

historical_valid = (
    ~np.ma.getmaskarray(
        historical
    )
)

forest2024_valid = (
    ~np.ma.getmaskarray(
        forest2024
    )
)

disturbance_valid = (
    ~np.ma.getmaskarray(
        disturbance
    )
)

drivers_valid = (
    ~np.ma.getmaskarray(
        drivers
    )
)

storm_valid = (
    ~np.ma.getmaskarray(
        storm
    )
)


# =============================================================================
# 1. TOTAL MANGROVE AREA
#
# User AOI
# -> ecosystem_v3 == 2
# =============================================================================

mangrove_mask = (
    ecosystem_valid
    & (
        ecosystem.data
        == MANGROVE_CLASS
    )
)

total_mangrove_area = area_ha(
    mangrove_mask,
    transform
)


# =============================================================================
# 2. REMAINING MANGROVE FOREST
#
# Mangrove mask
# -> historical_deforestation_v3 == 1
# =============================================================================

remaining_mask = (
    mangrove_mask
    & historical_valid
    & (
        historical.data
        == REMAINING_FOREST_CLASS
    )
)

remaining_area = area_ha(
    remaining_mask,
    transform
)


# =============================================================================
# 3. CURRENT MANGROVE FOREST
#
# Mangrove mask
# -> forest_2024_v3 == 1
# =============================================================================

current_mangrove_mask = (
    mangrove_mask
    & forest2024_valid
    & (
        forest2024.data
        == CURRENT_FOREST_CLASS
    )
)


# =============================================================================
# 4. DISTURBED MANGROVE
#
# Current mangrove
# -> forest_disturbance_v3 > 0
#
# This becomes the MASTER MASK for all driver analysis.
# =============================================================================

disturbed_mangrove_mask = (
    current_mangrove_mask
    & disturbance_valid
    & (
        disturbance.data
        > DISTURBANCE_THRESHOLD
    )
)

disturbed_area = area_ha(
    disturbed_mangrove_mask,
    transform
)


# =============================================================================
# PERCENTAGES
# =============================================================================

remaining_percentage = (
    remaining_area
    / total_mangrove_area
    * 100
    if total_mangrove_area > 0
    else 0
)

disturbed_percentage = (
    disturbed_area
    / total_mangrove_area
    * 100
    if total_mangrove_area > 0
    else 0
)


# =============================================================================
# 5. NON-NATURAL DRIVERS
#
# Disturbed mangrove
# -> drivers_disturbance_v3
#
# 1,2,3,4 = Commodities
# 5       = Settlement
# =============================================================================

commodities_mask = (
    disturbed_mangrove_mask
    & drivers_valid
    & np.isin(
        drivers.data,
        COMMODITY_CLASSES
    )
)

settlement_mask = (
    disturbed_mangrove_mask
    & drivers_valid
    & (
        drivers.data
        == SETTLEMENT_CLASS
    )
)


commodities_area = area_ha(
    commodities_mask,
    transform
)

settlement_area = area_ha(
    settlement_mask,
    transform
)


non_natural_drivers = []

if np.any(
    commodities_mask
):

    non_natural_drivers.append(
        "Commodities"
    )

if np.any(
    settlement_mask
):

    non_natural_drivers.append(
        "Settlement"
    )


# =============================================================================
# 6. MAIN PRESSURE
#
# Largest overlap area between:
# - Commodities
# - Settlement
# =============================================================================

if commodities_area > settlement_area:

    main_pressure = (
        "Commodities"
    )

elif settlement_area > commodities_area:

    main_pressure = (
        "Settlement"
    )

elif (
    commodities_area > 0
    and settlement_area > 0
):

    main_pressure = (
        "Commodities and Settlement"
    )

else:

    main_pressure = (
        "Not identified"
    )


# =============================================================================
# 7. NATURAL DRIVER
#
# Disturbed mangrove
# -> risk_storm_v3
# -> pixel 4 or 5
# =============================================================================

storm_mask = (
    disturbed_mangrove_mask
    & storm_valid
    & np.isin(
        storm.data,
        STORM_RISK_CLASSES
    )
)


natural_drivers = []

if np.any(
    storm_mask
):

    natural_drivers.append(
        "Extreme climate event"
    )


# =============================================================================
# 8. OTHER DRIVER
#
# Disturbed mangrove pixels that DO NOT overlap
# known drivers_disturbance_v3 classes 1-5.
#
# Natural storm risk is independent and does not remove
# a pixel from the "Other" classification.
# =============================================================================

known_driver_mask = (
    disturbed_mangrove_mask
    & drivers_valid
    & np.isin(
        drivers.data,
        [1, 2, 3, 4, 5]
    )
)

other_mask = (
    disturbed_mangrove_mask
    & ~known_driver_mask
)


other_drivers = []

if np.any(
    other_mask
):

    other_drivers.append(
        "Other"
    )


# =============================================================================
# DEBUG: DATA INTERACTION
#
# Useful during testing to identify where pixels disappear.
# =============================================================================

print()
print("=" * 75)
print("MANGROVE MASK CHECK")
print("=" * 75)

print(
    f"Mangrove pixels             : "
    f"{np.count_nonzero(mangrove_mask):,}"
)

print(
    f"Current mangrove pixels     : "
    f"{np.count_nonzero(current_mangrove_mask):,}"
)

print(
    f"Disturbed mangrove pixels   : "
    f"{np.count_nonzero(disturbed_mangrove_mask):,}"
)

print(
    f"Commodities pixels          : "
    f"{np.count_nonzero(commodities_mask):,}"
)

print(
    f"Settlement pixels           : "
    f"{np.count_nonzero(settlement_mask):,}"
)

print(
    f"Storm risk 4/5 pixels       : "
    f"{np.count_nonzero(storm_mask):,}"
)

print(
    f"Other pixels                : "
    f"{np.count_nonzero(other_mask):,}"
)

# =============================================================================
# BACKEND RESULT
# =============================================================================

result = {

    "mangrove": {

        "total_area_ha":
            round(
                total_mangrove_area,
                2
            ),

        "remaining_forest": {

            "area_ha":
                round(
                    remaining_area,
                    2
                ),

            "percentage":
                round(
                    remaining_percentage,
                    2
                )
        },

        "disturbed": {

            "area_ha":
                round(
                    disturbed_area,
                    2
                ),

            "percentage":
                round(
                    disturbed_percentage,
                    2
                )
        },

        "main_pressure":
            main_pressure,

        "drivers": {

            "non_natural":
                non_natural_drivers,

            "natural":
                natural_drivers,

            "other":
                other_drivers
        }
    }
}


# =============================================================================
# FINAL OUTPUT
# =============================================================================

print()
print("=" * 75)
print("MANGROVE DISTURBANCE")
print("=" * 75)

print(
    f"Total area                : "
    f"{result['mangrove']['total_area_ha']:,.2f} ha"
)

print(
    f"Remaining mangrove forest : "
    f"{result['mangrove']['remaining_forest']['area_ha']:,.2f} ha "
    f"({result['mangrove']['remaining_forest']['percentage']:.2f}%)"
)

print(
    f"Disturbed                 : "
    f"{result['mangrove']['disturbed']['area_ha']:,.2f} ha "
    f"({result['mangrove']['disturbed']['percentage']:.2f}%)"
)

print(
    f"Main pressure             : "
    f"{result['mangrove']['main_pressure']}"
)


print()
print("Mangrove disturbance drivers")
print("-" * 75)


print()
print("Non-natural drivers:")

if non_natural_drivers:

    for driver in non_natural_drivers:

        print(
            f"  - {driver}"
        )

else:

    print(
        "  None detected"
    )


print()
print("Natural drivers:")

if natural_drivers:

    for driver in natural_drivers:

        print(
            f"  - {driver}"
        )

else:

    print(
        "  None detected"
    )


print()
print("Other drivers:")

if other_drivers:

    for driver in other_drivers:

        print(
            f"  - {driver}"
        )

else:

    print(
        "  None detected"
    )

## 7.4 Peatland Ecosystem

Reports the drivers of peatland disturbance.

**Data.** `forest_drivers_v3.tif: pixel ranging from 1 to 11 ` This data comes from
Bart Slagter, et al 2026 https://doi.org/10.21203/rs.3.rs-7424252/v1
Which only focuses on classifying the key drivers of forest disturbances and 
may not include all potential causes of deforestation.


**Calibration warning.**  
The drainage canal dataset developed by Dadap et al. (2021) captures drainage canals located across both peatland and non-peatland areas. Therefore, the dataset was masked using `ecosystem_v3.tif` to retain only drainage canals occurring within mapped peatland areas in Southeast Asia.

Drainage pressure, however, refers to the proximity of drainage canals to the project area, including canals located outside the project boundary that may still influence peatland conditions within the project area. This follows previous findings indicating that:

a) Astiani et al. (2017) found that canals can influence water-table depth up to 500 m from the surrounding area.

b) Wedeux et al. (2020) found that canal networks may affect forest biomass growth up to 1 km away.

## Peatland Disturbance

| Indicator | Value |
|---|---:|
| **Total area** | **24,694.84 ha** |
| **Remaining peatland forest** | **1,182.02 ha (4.79%)** |
| **Disturbed** | **165.43 ha (0.67%)** |
| **Converted / Loss** | **4,968.08 ha (20.12%)** |

### Peatland Disturbance Drivers

| Driver | Status |
|---|---:|
| **Canal proximity** | **High** |
| **Nearest canal** | **0.00 m** |
| **Drainage pressure** | **High** |
| **Fire risk** | **High** |

In [ ]:
aoi = gpd.read_file(
    USER_AOI
)

if aoi.empty:
    raise ValueError(
        "AOI contains no features."
    )

if aoi.crs is None:
    raise ValueError(
        "AOI has no CRS."
    )

aoi = aoi[
    aoi.geometry.notna()
    & ~aoi.geometry.is_empty
].copy()


# =============================================================================
# READ RASTER
# =============================================================================

def read_raster(path):

    with rasterio.open(path) as src:

        polygon = (
            aoi.to_crs(src.crs)
            if aoi.crs != src.crs
            else aoi
        )

        geometries = [
            geom.__geo_interface__
            for geom in polygon.geometry
        ]

        data, transform = mask(
            src,
            geometries,
            crop=True,
            filled=True,
            nodata=0
        )

        return (
            data[0],
            transform,
            src.crs
        )

# =============================================================================
# AREA CALCULATION
# EPSG:4326 -> hectares
# =============================================================================

GEOD = Geod(
    ellps="WGS84"
)


def area_ha(
    binary_mask,
    transform
):

    binary_mask = np.asarray(
        binary_mask,
        dtype=bool
    )

    total_m2 = 0.0

    pixel_width = abs(
        transform.a
    )

    for row in range(
        binary_mask.shape[0]
    ):

        count = np.count_nonzero(
            binary_mask[row]
        )

        if count == 0:
            continue

        north = (
            transform.f
            + row * transform.e
        )

        south = (
            north
            + transform.e
        )

        west = transform.c
        east = west + pixel_width

        pixel_m2, _ = (
            GEOD.polygon_area_perimeter(
                [west, east, east, west],
                [north, north, south, south]
            )
        )

        total_m2 += (
            count
            * abs(pixel_m2)
        )

    return (
        total_m2 / 10000
    )


# =============================================================================
# LOAD RASTERS
# =============================================================================

ecosystem, transform, raster_crs = read_raster(
    ECOSYSTEM
)

historical, historical_transform, _ = read_raster(
    HISTORICAL
)

forest2024, forest2024_transform, _ = read_raster(
    FOREST_2024
)

disturbance, disturbance_transform, _ = read_raster(
    DISTURBANCE
)

canal_density, canal_transform, _ = read_raster(
    PEAT_CANALS_DENSITY
)

fire_risk, fire_transform, _ = read_raster(
    FIRE_RISK
)

drainage_canals, drainage_transform, drainage_crs = read_raster(
    DRAINAGE_CANALS
)

# =============================================================================
# 1. TOTAL PEATLAND
#
# AOI
# -> ecosystem_v3 == 3
# =============================================================================

peatland_mask = (
    ecosystem == PEATLAND
)

total_area = area_ha(
    peatland_mask,
    transform
)


# =============================================================================
# 2. REMAINING PEATLAND FOREST
#
# Peatland
# -> historical_deforestation_v3 == 1
# =============================================================================

remaining_mask = (
    peatland_mask
    & (
        historical
        == REMAINING_FOREST
    )
)

remaining_area = area_ha(
    remaining_mask,
    transform
)


# =============================================================================
# 3. CURRENT PEATLAND FOREST
#
# Peatland
# -> forest_2024_v3 == 1
# =============================================================================

current_peatland_mask = (
    peatland_mask
    & (
        forest2024
        == CURRENT_FOREST
    )
)


# =============================================================================
# 4. DISTURBED PEATLAND
#
# Current peatland forest
# -> forest_disturbance_v3 > 0
# =============================================================================

disturbed_mask = (
    current_peatland_mask
    & (
        disturbance
        > DISTURBANCE_THRESHOLD
    )
)

disturbed_area = area_ha(
    disturbed_mask,
    transform
)


# =============================================================================
# 5. CONVERTED / LOSS
#
# Peatland
# -> historical_deforestation_v3 == 2
# =============================================================================

loss_mask = (
    peatland_mask
    & (
        historical
        == FOREST_LOSS
    )
)

loss_area = area_ha(
    loss_mask,
    transform
)


# =============================================================================
# PERCENTAGES
# =============================================================================

def percentage(area):

    if total_area == 0:
        return 0

    return (
        area
        / total_area
        * 100
    )


remaining_percentage = percentage(
    remaining_area
)

disturbed_percentage = percentage(
    disturbed_area
)

loss_percentage = percentage(
    loss_area
)


# =============================================================================
# CANAL PROXIMITY
#
# drainage_canal_v3.tif
# pixel 1 = drainage canal
#
# Current peatland forest -> nearest canal
#
# <= 500 m       = High
# > 500-1000 m   = Medium
# > 1000 m       = Low
# =============================================================================

from rasterio.features import shapes
from shapely.geometry import shape


def mask_to_geometry(
    binary_mask,
    transform,
    crs
):

    geometries = [
        shape(geom)
        for geom, value in shapes(
            binary_mask.astype(
                np.uint8
            ),
            mask=binary_mask,
            transform=transform
        )
        if value == 1
    ]

    if not geometries:
        return None

    return (
        gpd.GeoSeries(
            geometries,
            crs=crs
        )
        .union_all()
    )


def canal_proximity_level():

    # Current peatland forest
    if not np.any(
        current_peatland_mask
    ):
        return "Not identified", None


    # Drainage canal pixel = 1
    canal_mask = (
        drainage_canals == 1
    )


    if not np.any(
        canal_mask
    ):
        return "Low", None


    # Convert raster masks to geometry
    peat_forest_geometry = mask_to_geometry(
        current_peatland_mask,
        transform,
        raster_crs
    )

    canal_geometry = mask_to_geometry(
        canal_mask,
        drainage_transform,
        drainage_crs
    )


    if (
        peat_forest_geometry is None
        or canal_geometry is None
    ):
        return "Low", None


    # Use projected CRS for metre distance
    metric_crs = (
        aoi.estimate_utm_crs()
    )


    peat_metric = (
        gpd.GeoSeries(
            [peat_forest_geometry],
            crs=raster_crs
        )
        .to_crs(metric_crs)
        .iloc[0]
    )


    canal_metric = (
        gpd.GeoSeries(
            [canal_geometry],
            crs=drainage_crs
        )
        .to_crs(metric_crs)
        .iloc[0]
    )


    # Nearest distance
    distance_m = (
        peat_metric.distance(
            canal_metric
        )
    )


    if distance_m <= 500:

        level = "High"

    elif distance_m <= 1000:

        level = "Medium"

    else:

        level = "Low"


    return (
        level,
        float(distance_m)
    )


canal_proximity, canal_distance_m = (
    canal_proximity_level()
)
# =============================================================================
# 7. DRAINAGE PRESSURE
#
# Peatland
# -> peat_canals_density_v3
#
# 3 = High
# 2 = Medium
# 1 = Low
#
# UI uses highest class present.
# =============================================================================

drainage_values = np.unique(
    canal_density[
        peatland_mask
    ]
)


if 3 in drainage_values:

    drainage_pressure = (
        "High"
    )

elif 2 in drainage_values:

    drainage_pressure = (
        "Medium"
    )

elif 1 in drainage_values:

    drainage_pressure = (
        "Low"
    )

else:

    drainage_pressure = (
        "Not identified"
    )


# =============================================================================
# 8. FIRE RISK
#
# Peatland
# -> fire_risk_v3
#
# 4 = High
# 3 = Medium
# 2 = Low
# 1 = Very low
# 0 / NoData = No risk
#
# UI uses highest class present.
# =============================================================================

fire_values = np.unique(
    fire_risk[
        peatland_mask
    ]
)


if 4 in fire_values:

    fire_risk_level = (
        "High"
    )

elif 3 in fire_values:

    fire_risk_level = (
        "Medium"
    )

elif 2 in fire_values:

    fire_risk_level = (
        "Low"
    )

elif 1 in fire_values:

    fire_risk_level = (
        "Very low"
    )

else:

    fire_risk_level = (
        "No risk"
    )


# =============================================================================
# BACKEND RESULT
# =============================================================================

result = {

    "peatland": {

        "total_area_ha":
            round(
                total_area,
                2
            ),

        "remaining_forest": {

            "area_ha":
                round(
                    remaining_area,
                    2
                ),

            "percentage":
                round(
                    remaining_percentage,
                    2
                )
        },

        "disturbed": {

            "area_ha":
                round(
                    disturbed_area,
                    2
                ),

            "percentage":
                round(
                    disturbed_percentage,
                    2
                )
        },

        "converted_loss": {

            "area_ha":
                round(
                    loss_area,
                    2
                ),

            "percentage":
                round(
                    loss_percentage,
                    2
                )
        },

        "drivers": {

            "canal_proximity":
                canal_proximity,

            "canal_distance_m":
                (
                    round(
                        canal_distance_m,
                        2
                    )
                    if canal_distance_m
                    is not None
                    else None
                ),

            "drainage_pressure":
                drainage_pressure,

            "fire_risk":
                fire_risk_level
        }
    }
}


# =============================================================================
# FINAL OUTPUT
# =============================================================================

r = result[
    "peatland"
]


print()
print("=" * 75)
print("PEATLAND DISTURBANCE")
print("=" * 75)


print(
    f"Total area                : "
    f"{r['total_area_ha']:,.2f} ha"
)


print(
    f"Remaining peatland forest : "
    f"{r['remaining_forest']['area_ha']:,.2f} ha "
    f"({r['remaining_forest']['percentage']:.2f}%)"
)


print(
    f"Disturbed                 : "
    f"{r['disturbed']['area_ha']:,.2f} ha "
    f"({r['disturbed']['percentage']:.2f}%)"
)


print(
    f"Converted / Loss          : "
    f"{r['converted_loss']['area_ha']:,.2f} ha "
    f"({r['converted_loss']['percentage']:.2f}%)"
)


print()
print("Peatland disturbance drivers")
print("-" * 75)


print(
    f"Canal proximity  : "
    f"{r['drivers']['canal_proximity']}"
)


if (
    r["drivers"]["canal_distance_m"]
    is not None
):

    print(
        f"Nearest canal    : "
        f"{r['drivers']['canal_distance_m']:,.2f} m"
    )


print(
    f"Drainage pressure: "
    f"{r['drivers']['drainage_pressure']}"
)


print(
    f"Fire risk        : "
    f"{r['drivers']['fire_risk']}"
)

---
## Save

Each section above already ran and displayed itself. This cell writes them to one combined JSON.

In [ ]:
path = save_results(results, aoi, aoi_id, STAGE_GENERAL)
print(f"Saved {path}")